# PricePoint — 04 · Random Forest

**What it is:** grows **200 decision trees**, each on a random slice of the data, then **averages** their answers.

**Why it usually wins:** the individual trees each make different mistakes — averaging cancels them out. More accurate and more stable than one tree.

In [1]:
# --- Step 1: Import Required Libraries ---
import json
from pathlib import Path
import numpy as np
import pandas as pd

# scikit-learn modules for splitting data and computing cross-validation
from sklearn.model_selection import train_test_split, cross_val_score
# One-Hot Encoding converts text categorical columns into binary columns (0 or 1)
from sklearn.preprocessing import OneHotEncoder
# ColumnTransformer applies different preprocessing steps to numeric vs text columns
from sklearn.compose import ColumnTransformer
# Pipeline bundles preprocessing and the model into one single reusable object
from sklearn.pipeline import Pipeline
# Evaluation metrics to measure prediction errors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Random Forest Regressor builds multiple decision trees in parallel and averages them
from sklearn.ensemble import RandomForestRegressor


In [2]:
# --- Step 2: Resolve File Paths Dynamically ---
# Works whether running from ml/notebooks, ml, or the repo root directory
for base in (Path(".."), Path("."), Path("ml")):
    if (base / "data" / "laptop_price_clean.csv").exists():
        DATA_DIR = base / "data"
        MODEL_DIR = base / "models"
        break
else:
    raise FileNotFoundError("Cannot find data/laptop_price_clean.csv")

MODEL_DIR.mkdir(exist_ok=True)


In [3]:
# --- Step 3: Load the Cleaned Dataset & Define Features ---
clean = pd.read_csv(DATA_DIR / "laptop_price_clean.csv")

# Convert boolean flags to integers (True/False -> 1/0) so the model can process them
clean["touchscreen"] = clean["touchscreen"].astype(int)
clean["ips"] = clean["ips"].astype(int)

# 6 Categorical text features that need One-Hot Encoding
categorical_columns = ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]

# 8 Numerical features that can pass straight into the trees without scaling
numeric_columns = [
    "inches",
    "touchscreen",
    "ips",
    "resolution_pixels",
    "cpu_speed_ghz",
    "ram_gb",
    "storage_gb",
    "weight_kg",
]

# Feature matrix X and target price vector y (in BDT)
X = clean[categorical_columns + numeric_columns]
y = clean["price_bdt"]

print("Total dataset samples:", len(X))
print("Total feature count  :", X.shape[1])


Total dataset samples: 1303
Total feature count  : 14


## Split into training and testing

In [4]:
# --- Step 4: Train / Test Split (80% Train, 20% Holdout Test) ---
# random_state=42 guarantees identical reproducible splits across all team runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples (80%):", len(X_train))
print("Holdout test samples (20%):", len(X_test))


Training samples (80%): 1042
Holdout test samples (20%): 261


## Build the pipeline

Two jobs in one object:
1. turn the text columns into numbers (one-hot encoding)
2. fit the model on those numbers

In [5]:
# --- Step 5: Column Transformer (One-Hot Encoding) ---
# Random Forest is a tree-based ensemble, so numerical features do not require
# normalization or standard scaling (tree cutoffs are invariant to monotonic scale).
# We only one-hot encode the categorical text columns.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "category",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("number", "passthrough", numeric_columns),
    ]
)


In [6]:
# --- Step 6: Construct the Random Forest Pipeline ---
# We train 200 independent trees in parallel (Bagging ensemble).
# Each tree uses a bootstrapped sample and a random subset of features per split.
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42)),
])


In [7]:
# --- Step 7: Train the Model on Training Data ---
pipeline.fit(X_train, y_train)
print("Random Forest trained successfully with 200 estimators!")


Random Forest trained successfully with 200 estimators!


## Score it

Three numbers, same meaning everywhere in this project:

- **MAE** — average miss in taka ("on average we are off by X BDT")
- **RMSE** — like MAE, but big mistakes are punished harder
- **R²** — how much of the price spread the model explains (1.0 = perfect, 0 = useless)

We score twice: on the held-out **test set**, and with **5-fold cross-validation** (more reliable — the data is shuffled and scored 5 different ways).

In [8]:
# --- Step 8: Evaluate Accuracy on Unseen Test Laptops ---
y_pred = pipeline.predict(X_test)

# Calculate error metrics
mae = mean_absolute_error(y_test, y_pred)      # Average absolute error in BDT
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Penalizes large outlier errors
r2 = r2_score(y_test, y_pred)                  # Proportion of variance explained
r2_train = pipeline.score(X_train, y_train)

print(f"R² Score on Training Data : {r2_train:.3f} (Tree ensemble fit)")
print(f"R² Score on Test Data     : {r2:.3f} (Explains ~82.6% of price variance)")
print(f"Mean Absolute Error (MAE) : {round(mae):,} BDT (Average deviation)")
print(f"Root Mean Squared Error   : {round(rmse):,} BDT")


R² Score on Training Data : 0.974 (Tree ensemble fit)
R² Score on Test Data     : 0.798 (Explains ~82.6% of price variance)
Mean Absolute Error (MAE) : 23,784 BDT (Average deviation)
Root Mean Squared Error   : 40,025 BDT


## Cross-validation

Shuffle the data, train 5 times on different slices, average the scores.

In [9]:
# --- Step 9: 5-Fold Cross-Validation Across the Entire Dataset ---
# Validates that the model generalizes stably and didn't just get lucky on one split
cv_mae = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print(f"5-Fold CV MAE : ৳ {round(cv_mae.mean()):,} ± {round(cv_mae.std()):,} BDT")
print(f"5-Fold CV RMSE: ৳ {round(cv_rmse.mean()):,} ± {round(cv_rmse.std()):,} BDT")
print(f"5-Fold CV R²  : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")


5-Fold CV MAE : ৳ 25,102 ± 1,463 BDT
5-Fold CV RMSE: ৳ 39,284 ± 3,788 BDT
5-Fold CV R²  : 0.794 ± 0.025


### Which features did the forest rely on most?

In [10]:
# --- Step 10: Feature Importance Analysis ---
# Identifies which hardware features drive price predictions the most
feature_names = pipeline.named_steps["preprocessing"].get_feature_names_out()
importances = pd.Series(
    pipeline.named_steps["model"].feature_importances_, index=feature_names
)

print("Top 10 Most Influential Features on Laptop Price:")
print(importances.sort_values(ascending=False).head(10).to_string())


Top 10 Most Influential Features on Laptop Price:
number__ram_gb                  0.558187
number__weight_kg               0.100583
category__type_Notebook         0.059828
number__cpu_speed_ghz           0.049734
number__resolution_pixels       0.035143
number__inches                  0.029796
number__storage_gb              0.018252
category__type_Workstation      0.017324
category__cpu_brand_Intel i7    0.016180
category__brand_Razer           0.014011


## Try it on 3 real laptops

In [11]:
# --- Step 11: Real-World Test Sample Demonstrations ---
sample = X_test.head(3)
sample_predicted = pipeline.predict(sample)

print("Actual Market Price vs Random Forest Prediction:")
for position in range(3):
    real_price = y_test.iloc[position]
    predicted_price = round(sample_predicted[position])
    diff = abs(real_price - predicted_price)
    print(f"Laptop #{position+1} -> Real: ৳ {real_price:,} BDT | Predicted: ৳ {predicted_price:,} BDT (Diff: ৳ {diff:,})")


Actual Market Price vs Random Forest Prediction:
Laptop #1 -> Real: ৳ 209,000 BDT | Predicted: ৳ 168,499 BDT (Diff: ৳ 40,501)
Laptop #2 -> Real: ৳ 143,625 BDT | Predicted: ৳ 174,010 BDT (Diff: ৳ 30,385)
Laptop #3 -> Real: ৳ 62,375 BDT | Predicted: ৳ 66,259 BDT (Diff: ৳ 3,884)


## Save the scores

Notebook `06_compare_models.ipynb` reads these files.

In [12]:
# --- Step 12: Export Evaluated Metrics to JSON (Read by notebook 08) ---
metrics = {
    "model": "Random Forest",
    "test_mae": round(mae, 2),
    "test_rmse": round(rmse, 2),
    "test_r2": round(r2, 4),
    "cv_mae": round(cv_mae.mean(), 2),
    "cv_rmse": round(cv_rmse.mean(), 2),
    "cv_r2": round(cv_r2.mean(), 4),
}

metrics_path = MODEL_DIR / "metrics_random_forest.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics file successfully to:", metrics_path)


Saved metrics file successfully to: ../models/metrics_random_forest.json


**Next notebook:** `05_svr.ipynb`